In [52]:
import os, re, json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
solar_api = os.getenv("SOLAR_API")

client = OpenAI(
    api_key=solar_api,
    base_url="https://api.upstage.ai/v1"
)


In [53]:
content = """


2. 주요 제품 및 서비스



(1). 주요 제품 등의 현황
                                                                                                  (단위 :백만원) 

사업부문	매출유형	품   목	구체적용도	주요상표등	매출액	비율(%)
인쇄 및
제조	내  수	계획생산	문구류	-	유즈어리,
퍼스널,
Window외	12,488	22.35%
주문생산	인쇄및제본	-	18,879	33.79%
수   출	주문생산	인쇄및제본	-	15,289	27.37%
 상품매출	내   수	기프트몰	판촉물류	-	-	150	0.27%
위탁경영	문구류	-	-	2,709	4.85%
디지털기기류	-	-	6,322	11.32%
부동산임대	내   수	-	임대	-	-	27	0.05%
합   계	55,864	100.00%

(2). 주요제품 등의 가격변동 추이
인쇄사업 특성 상 비교적 다품종 소량생산의 경우가 많아 제품별 판매가격 추이를 산정하기 어렵고, 주요 제품 등의 가격변동원인은 주요 원재료 매입가격의 변동과 거래처의 요구 스펙 및 수량등에 따라 상이한 가격으로 판매되기 때문입니다.






"""

In [54]:
prompt = """
너는 기업 사업보고서의 “2. 주요 제품 및 서비스”에서 (가) ‘주요 제품 등의 현황’에 해당하는 부분을 바탕으로
핵심 제품/서비스 후보와 짧은 기능(용도) 설명을 추출한다.

다른 문장/표/해설은 금지한다.

출력은 반드시 아래 형식의 텍스트로만 한다. JSON 금지.

[출력 형식]
핵심 제품
    • 제품명 : 짧은 기능 설명
    • 제품명 : 짧은 기능 설명
    • 제품명 : 짧은 기능 설명


[규칙]
- 제품/서비스 후보는 ‘품목(또는 유사한 제품명 항목)’에서 추출한다.
- “사업부문/내수/수출/계획생산/주문생산/단위/합계” 등은 제품명이 아니다.
- 후보가 애매하면 ‘제품/서비스명’만 간단히 적고, desc는 12~25자 내외로 기능/용도 중심으로 쓴다.
- 브랜드명, 회사명은 desc에 넣지 마라.
- 중복 후보는 통합하라.
- 최대 10개 후보까지 가능(후처리에서 5개로 자름). 억지로 채우지 마라.


""".strip()


In [55]:
def slice_ga_section(text: str) -> str:
    """
    기업마다 (나) 표기가 달라서 '가격변동' 계열/ (2) / 나. 등을 폭넓게 커트.
    (가) 현황 중심 텍스트만 남기는 범용 전처리.
    """
    t = text or ""

    cut_markers = [
        "나. 주요", "나. 주 요", "(2). 주요", "(2) 주요", "2). 주요", "2) 주요",
        "주요제품 등의 가격", "주요 제품 등의 가격", "가격변동", "가격 변동",
        "가격변동추이", "가격변동 추이"
    ]
    idxs = [t.find(m) for m in cut_markers if t.find(m) != -1]
    if idxs:
        t = t[:min(idxs)]

    return t.strip()


# ✅ 코스닥 전체에서 '제품'이 아닌 수익항목을 최대한 잘 거르는 금칙어(필요시 확장)
DENY_NAME_PATTERNS = [
    r"부동산", r"임대", r"리스",
    r"이자", r"배당",
    r"처분", r"평가", r"환차", r"파생",
    r"수수료", r"로열티",
    r"기타", r"합\s*계", r"합계",
    r"A\s*/?\s*S|부품|유지보수|수리|서비스부품"

]

# 표에서 자주 튀어나오는 '제품명이 아닌 단어'
DENY_EXACT = {
    "-", "내수", "수출", "계획생산", "주문생산", "계", "합", "합계"
}

def norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").replace("\u3000", " ")).strip()

def match_any(patterns, text: str) -> bool:
    for p in patterns:
        if re.search(p, text, flags=re.IGNORECASE):
            return True
    return False

def postprocess_to_bullets(model_text: str, max_items: int = 5) -> str:
    """
    LLM이 JSON이 아닌 '텍스트 bullet'을 출력하도록 했을 때,
    그 텍스트에서 제품명만 추려서(금칙어/중복 제거) 표준 bullet 포맷으로 정리.
    """
    t = (model_text or "").strip()
    if not t:
        return "핵심 제품\n    • (추출 실패) : 빈 응답"

    lines = [norm(x) for x in t.splitlines() if norm(x)]
    out = ["핵심 제품"]
    seen = set()

    # 1) LLM이 이미 "• 제품명 : 설명" 형태로 준다고 가정하고 파싱
    for line in lines:
        # 헤더 라인 제거
        if re.fullmatch(r"핵심\s*제품", line):
            continue

        # bullet 기호 제거(•, -, *)
        line2 = re.sub(r"^[\-\*\u2022]\s*", "", line).strip()

        # "제품명 : 설명" 형태가 아니면 스킵
        if ":" not in line2:
            continue

        name, desc = [norm(x) for x in line2.split(":", 1)]

        if not name:
            continue
        if name in DENY_EXACT:
            continue
        if match_any(DENY_NAME_PATTERNS, name):
            continue

        key = re.sub(r"\s+", "", name)
        if key in seen:
            continue
        seen.add(key)

        if not desc:
            desc = "핵심 제품/서비스"

        out.append(f"    • {name} : {desc}")

        if len(seen) >= max_items:
            break

    if len(out) == 1:
        out.append("    • (추출 실패) : 유효 후보 없음")

    return "\n".join(out)


In [56]:
content_for_llm = slice_ga_section(content)

response = client.chat.completions.create(
    model="solar-pro3",
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": content_for_llm}
    ],
    temperature=0,
    reasoning_effort="low",
    max_tokens=1024,

)

result = postprocess_to_bullets(response.choices[0].message.content, max_items=3)
print(result)


핵심 제품
    • 문구류 : 사무용품 및 개인용 문구 제품
    • 인쇄및제본 : 주문형 인쇄 및 제본 서비스
    • 기프트몰 : 판촉물 물류 및 선물 상품
